In [6]:
import pandas as pd
import numpy as np

# Loading the preprocessed file
dataset1 = pd.read_csv("Kokila_Rubini.csv", index_col=None) 
print("Dataset loaded successfully. Shape:", dataset1.shape)
print("Columns:", dataset1.columns.tolist())

# Preparing features (indep_X) and target (dep_Y)
df2 = dataset1.copy()
df2 = pd.get_dummies(df2, drop_first=True)

# Defining independent features and dependent target (Flirting_Encoded)
indep_X = df2.drop(columns=["Flirting_Encoded"])
dep_Y = df2["Flirting_Encoded"]

Dataset loaded successfully. Shape: (800, 12)
Columns: ['Date', 'Time', 'Name', 'Chat', 'scores', 'compound', 'Negtive', 'Postive', 'Neutral', 'comp_score', 'Topic', 'Flirting_Encoded']


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif  # Changed from chi2 to f_classif
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Loading preprocessed file
dataset1 = pd.read_csv("Kokila_Rubini.csv", index_col=None)
df2 = dataset1.copy()

# Converting categorical variables into dummy/indicator variables
df2 = pd.get_dummies(df2, drop_first=True)

# Selecting only numerical columns for independent variables to avoid text dtype issues in SelectKBest
indep_X = df2.select_dtypes(include=['number']).drop(columns=["Flirting_Encoded"], errors='ignore')
dep_Y = df2["Flirting_Encoded"]

# 2. Feature Selection using SelectKBest with f_classif (supports negative sentiment values)
def selectkbest(indep_X, dep_Y, n):
    # Using f_classif instead of chi2 to handle negative sentiment scores
    test = SelectKBest(score_func=f_classif, k=min(n, indep_X.shape[1]))
    fit1 = test.fit(indep_X, dep_Y)
    selectk_features = fit1.transform(indep_X)
    
    # Display feature scores
    feature_scores = pd.DataFrame({'Feature': indep_X.columns, 'Score': fit1.scores_})
    print("\n=== Feature Selection Scores (ANOVA F-Value) ===")
    print(feature_scores.sort_values(by='Score', ascending=False))
    
    return selectk_features

kbest = selectkbest(indep_X, dep_Y, 5)

# 3. Split & Scale
X_train, X_test, y_train, y_test = train_test_split(kbest, dep_Y, test_size=0.25, random_state=0)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# 4. Train and evaluate models
models = {
    'Logistic Regression': LogisticRegression(random_state=0),
    'SVM Linear': SVC(kernel='linear', random_state=0),
    'SVM Non-Linear': SVC(kernel='rbf', random_state=0),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(criterion='entropy', random_state=0),
    'Random Forest': RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
}

best_model = None
best_acc = 0.0

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_model = model

print(f"\nBest Performing Model selected with accuracy: {best_acc:.4f}")


=== Feature Selection Scores (ANOVA F-Value) ===
    Feature      Score
0  compound  74.371112
4     Topic  10.253888
1   Negtive   0.362798
2   Postive   0.324733
3   Neutral   0.089912
Logistic Regression Accuracy: 0.9950
SVM Linear Accuracy: 0.9950
SVM Non-Linear Accuracy: 0.9950
KNN Accuracy: 1.0000
Naive Bayes Accuracy: 0.8700
Decision Tree Accuracy: 0.9850
Random Forest Accuracy: 0.9950

Best Performing Model selected with accuracy: 1.0000


In [8]:
import pickle

# Saving the trained model and scaler
filename = "finalized_model_NLP.sav"
pickle.dump(best_model, open(filename, "wb"))
pickle.dump(sc, open("scaler.sav", "wb"))
print(f"Model saved successfully as {filename}")

# Loading test
loaded_model = pickle.load(open(filename, "rb"))
loaded_scaler = pickle.load(open("scaler.sav", "rb"))
print("Model loaded successfully from disk.")

Model saved successfully as finalized_model_NLP.sav
Model loaded successfully from disk.
